# Aula 2 — Classificação de imagens com CNN e ResNet

Na aula anterior, uma MLP recebeu features numéricas prontas. Agora construiremos uma rede capaz de **aprender as features diretamente dos pixels**. Primeiro implementaremos uma CNN pequena; depois entenderemos conexões residuais e aplicaremos transfer learning com uma ResNet-18.

> Este notebook é uma demonstração educacional e não constitui um sistema de diagnóstico médico.

## Por que não usar apenas uma MLP?

Uma imagem RGB de `224 × 224` possui `3 × 224 × 224 = 150.528` valores. Uma camada linear que ignorasse a organização espacial teria muitos parâmetros e trataria pixels vizinhos como posições independentes.

Uma CNN preserva a estrutura espacial e reutiliza pequenos filtros em toda a imagem. O mesmo filtro pode detectar uma borda ou textura em diferentes posições.

## Convolução

Um **kernel** é uma pequena matriz de pesos que percorre a imagem. Em cada posição, ele combina os pixels de uma região local e produz um valor em um novo mapa de características:

$$z_{i,j} = \sum_{c,u,v} K_{c,u,v} X_{c,i+u,j+v} + b$$

Os pesos do kernel são aprendidos por backpropagation, assim como os pesos das camadas lineares da MLP. As primeiras camadas costumam responder a padrões simples, enquanto camadas posteriores combinam esses padrões em representações mais abstratas.

### ReLU, pooling e canais

- **ReLU** adiciona não linearidade.
- **Pooling** reduz as dimensões espaciais, preservando respostas importantes.
- **Canais** representam diferentes mapas de características aprendidos.
- **Batch normalization** estabiliza as ativações e facilita o treinamento.

Nossa CNN aumentará os canais de `3 → 16 → 32 → 64` enquanto reduz altura e largura.

## Bibliotecas, seed e dispositivo

In [ ]:
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.preprocessing import label_binarize
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Dispositivo:", device)
print("CUDA:", torch.version.cuda)

## Manifesto preparado

O notebook [`01-prepare-images.pt-br.ipynb`](01-prepare-images.pt-br.ipynb) criou um manifesto com caminhos, rótulos e splits. O conjunto de teste já está definido, mas só será usado na avaliação final.

In [ ]:
def find_project_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Raiz do projeto não encontrada.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
MANIFEST_PATH = DATA_DIR / "alzheimer_mri_manifest.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "lesson-02"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not MANIFEST_PATH.exists():
    raise FileNotFoundError("Execute 01-prepare-images.pt-br.ipynb antes deste notebook.")

manifest = pd.read_csv(MANIFEST_PATH)
CLASS_NAMES = (
    manifest[["label", "class_name"]]
    .drop_duplicates().sort_values("label")["class_name"].tolist()
)
NUM_CLASSES = len(CLASS_NAMES)
pd.crosstab(manifest["split"], manifest["class_name"])

## Transformações

As imagens de treino recebem pequenas variações aleatórias de recorte e rotação. Isso é **data augmentation**: novas versões plausíveis são geradas durante o treino, sem alterar os arquivos originais.

Validação e teste usam transformações determinísticas. Os valores de normalização são os mesmos usados nos pesos ImageNet da ResNet, permitindo que CNN e ResNet recebam entradas equivalentes.

In [ ]:
IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.90, 1.00)),
    transforms.RandomRotation(degrees=5),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

evaluation_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

## Dataset com carregamento lazy

O objeto armazena somente o manifesto. Cada imagem é aberta em `__getitem__`, no momento em que seu batch é solicitado. Isso economiza memória e permite que o augmentation seja sorteado novamente a cada época.

In [ ]:
class AlzheimerImageDataset(Dataset):
    def __init__(self, frame, data_dir, transform):
        self.frame = frame.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        record = self.frame.iloc[index]
        image_path = self.data_dir / record["relative_path"]
        with Image.open(image_path) as image:
            image = image.convert("RGB")
            image = self.transform(image)
        label = torch.tensor(int(record["label"]), dtype=torch.long)
        return image, label


train_frame = manifest.query("split == 'train'").copy()
val_frame = manifest.query("split == 'validation'").copy()
test_frame = manifest.query("split == 'test'").copy()

train_dataset = AlzheimerImageDataset(train_frame, DATA_DIR, train_transform)
val_dataset = AlzheimerImageDataset(val_frame, DATA_DIR, evaluation_transform)
test_dataset = AlzheimerImageDataset(test_frame, DATA_DIR, evaluation_transform)

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 0  # configuração segura para notebooks no Windows e no Linux
generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), generator=generator,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

images, labels = next(iter(train_loader))
print("Batch de imagens:", images.shape)
print("Batch de rótulos:", labels.shape)

## Desbalanceamento das classes

Sem correção, erros na classe rara contribuiriam pouco para a loss total. Usaremos pesos inversamente proporcionais à frequência de cada classe no treino. Assim, um erro em uma classe rara recebe maior importância.

In [ ]:
class_counts = train_frame["label"].value_counts().sort_index()
class_weights = len(train_frame) / (NUM_CLASSES * class_counts.to_numpy())
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

pd.DataFrame({
    "classe": CLASS_NAMES,
    "amostras_treino": class_counts.to_numpy(),
    "peso_na_loss": class_weights.cpu().numpy(),
})

## CNN simples

A primeira arquitetura é pequena o suficiente para ser compreendida por inteiro:

```text
Imagem: 3 × 224 × 224
  ↓ Conv 3→16 + BatchNorm + ReLU + MaxPool
  ↓ Conv 16→32 + BatchNorm + ReLU + MaxPool
  ↓ Conv 32→64 + BatchNorm + ReLU + MaxPool
  ↓ Adaptive Average Pooling
  ↓ Linear 64→4
```

O `AdaptiveAvgPool2d` resume cada canal em um único valor. Isso evita uma camada linear enorme e permite receber imagens de dimensões espaciais variadas.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        return self.classifier(x)


def parameter_counts(model):
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    return total, trainable


simple_cnn = SimpleCNN(NUM_CLASSES)
total, trainable = parameter_counts(simple_cnn)
print(simple_cnn)
print(f"Parâmetros totais: {total:,}")
print(f"Parâmetros treináveis: {trainable:,}")

### Logits e CrossEntropyLoss

Assim como na MLP, a última camada produz quatro logits. A `CrossEntropyLoss` aplica internamente as operações necessárias para comparar esses scores com o rótulo correto. O `Softmax` será usado somente na avaliação, quando quisermos interpretar probabilidades.

## Treinamento reutilizável

O mesmo ciclo será usado para a CNN e a ResNet: treino, validação, armazenamento do melhor estado e early stopping. A validação orienta o treinamento; o teste ainda não é acessado.

In [ ]:
def keep_frozen_batchnorm_in_eval(model):
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            parameters = list(module.parameters())
            if parameters and not any(parameter.requires_grad for parameter in parameters):
                module.eval()


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    if training:
        model.train()
        keep_frozen_batchnorm_in_eval(model)
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for inputs, targets in loader:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            logits = model(inputs)
            loss = criterion(logits, targets)
            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * targets.size(0)
        total_correct += (logits.argmax(dim=1) == targets).sum().item()
        total_samples += targets.size(0)

    return total_loss / total_samples, total_correct / total_samples


def train_model(model, optimizer, epochs, name, patience=4):
    model = model.to(device)
    history = {"train_loss": [], "val_loss": [], "train_accuracy": [], "val_accuracy": []}
    best_state = deepcopy(model.state_dict())
    best_val_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        train_loss, train_accuracy = run_epoch(model, train_loader, optimizer)
        val_loss, val_accuracy = run_epoch(model, val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_accuracy"].append(val_accuracy)

        print(
            f"{name} | época {epoch:02d}/{epochs} | "
            f"loss {train_loss:.4f}/{val_loss:.4f} | "
            f"acurácia {train_accuracy:.2%}/{val_accuracy:.2%}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print("Early stopping: a loss de validação parou de melhorar.")
                break

    model.load_state_dict(best_state)
    torch.save(best_state, OUTPUT_DIR / f"{name}.pt")
    return model, history, best_val_loss


def plot_history(history, title):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, history["train_loss"], label="Treino")
    axes[0].plot(epochs, history["val_loss"], label="Validação")
    axes[0].set_title("Loss")
    axes[1].plot(epochs, history["train_accuracy"], label="Treino")
    axes[1].plot(epochs, history["val_accuracy"], label="Validação")
    axes[1].set_title("Acurácia")
    for axis in axes:
        axis.set_xlabel("Época")
        axis.grid(alpha=0.3)
        axis.legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
cnn_optimizer = torch.optim.AdamW(simple_cnn.parameters(), lr=1e-3, weight_decay=1e-4)
simple_cnn, cnn_history, cnn_best_val_loss = train_model(
    simple_cnn, cnn_optimizer, epochs=15, name="simple_cnn", patience=4
)
plot_history(cnn_history, "CNN simples")

## Da CNN para a ResNet

Adicionar camadas nem sempre melhora uma rede. Em arquiteturas profundas, o sinal e os gradientes atravessam muitas transformações, e o treinamento pode ficar mais difícil.

Uma conexão residual cria um caminho direto:

$$y = F(x) + x$$

Em vez de aprender toda a transformação desejada, o bloco pode aprender apenas uma **correção residual** `F(x)`. Se nenhuma correção for necessária, aproximar `F(x) = 0` preserva a entrada pelo atalho.

## Transfer learning com ResNet-18

A ResNet-18 foi pré-treinada no ImageNet. Suas primeiras camadas já aprenderam filtros para bordas, contrastes e texturas. Embora ressonâncias sejam diferentes de fotografias, essas representações iniciais podem ser reutilizadas.

Na primeira fase congelaremos o backbone e treinaremos apenas uma nova camada final com quatro saídas. Isso é **extração de features**, não treinamento completo da ResNet.

In [ ]:
resnet = models.resnet18(weights=ResNet18_Weights.DEFAULT)
for parameter in resnet.parameters():
    parameter.requires_grad = False

resnet.fc = nn.Linear(resnet.fc.in_features, NUM_CLASSES)
total, trainable = parameter_counts(resnet)
print(f"Parâmetros totais: {total:,}")
print(f"Parâmetros treináveis na fase 1: {trainable:,}")
print(f"Proporção treinável: {trainable / total:.3%}")

In [ ]:
head_optimizer = torch.optim.AdamW(resnet.fc.parameters(), lr=1e-3, weight_decay=1e-4)
resnet, head_history, head_best_val_loss = train_model(
    resnet, head_optimizer, epochs=8, name="resnet18_head", patience=3
)
head_state = deepcopy(resnet.state_dict())
plot_history(head_history, "ResNet-18 — treinamento da cabeça")

### Fine-tuning do último bloco

Agora descongelaremos somente o último bloco residual (`layer4`). Usaremos learning rates menores para adaptar representações de alto nível sem modificar bruscamente os filtros pré-treinados. Se a validação piorar, manteremos o estado obtido na fase anterior.

In [ ]:
RUN_FINE_TUNING = True

if RUN_FINE_TUNING:
    for parameter in resnet.layer4.parameters():
        parameter.requires_grad = True

    fine_tuning_optimizer = torch.optim.AdamW([
        {"params": resnet.layer4.parameters(), "lr": 1e-4},
        {"params": resnet.fc.parameters(), "lr": 5e-4},
    ], weight_decay=1e-4)

    resnet, fine_history, fine_best_val_loss = train_model(
        resnet, fine_tuning_optimizer, epochs=5, name="resnet18_finetuned", patience=3
    )

    if fine_best_val_loss > head_best_val_loss:
        print("O fine-tuning não melhorou a validação; restaurando a melhor cabeça.")
        resnet.load_state_dict(head_state)
    else:
        plot_history(fine_history, "ResNet-18 — fine-tuning do layer4")

## Avaliação final no teste

Somente agora os dois modelos recebem o conjunto de teste. Além da acurácia, usaremos métricas macro, que atribuem a mesma importância a cada classe, e balanced accuracy, adequada ao desbalanceamento.

In [ ]:
def evaluate_model(model, loader):
    model.eval()
    targets, predictions, probabilities = [], [], []

    with torch.no_grad():
        for inputs, batch_targets in loader:
            logits = model(inputs.to(device, non_blocking=True))
            batch_probabilities = torch.softmax(logits, dim=1)
            targets.extend(batch_targets.numpy())
            predictions.extend(batch_probabilities.argmax(dim=1).cpu().numpy())
            probabilities.extend(batch_probabilities.cpu().numpy())

    targets = np.asarray(targets)
    predictions = np.asarray(predictions)
    probabilities = np.asarray(probabilities)
    binary_targets = label_binarize(targets, classes=np.arange(NUM_CLASSES))

    metrics = {
        "Acurácia": accuracy_score(targets, predictions),
        "Acurácia balanceada": balanced_accuracy_score(targets, predictions),
        "Precisão macro": precision_score(targets, predictions, average="macro", zero_division=0),
        "Recall macro": recall_score(targets, predictions, average="macro", zero_division=0),
        "F1 macro": f1_score(targets, predictions, average="macro", zero_division=0),
        "ROC AUC macro": roc_auc_score(
            binary_targets, probabilities, average="macro", multi_class="ovr"
        ),
    }
    return metrics, targets, predictions, probabilities


cnn_metrics, test_targets, cnn_predictions, cnn_probabilities = evaluate_model(simple_cnn, test_loader)
resnet_metrics, _, resnet_predictions, resnet_probabilities = evaluate_model(resnet, test_loader)

comparison = pd.DataFrame([cnn_metrics, resnet_metrics], index=["CNN simples", "ResNet-18"])
comparison.round(3)

In [ ]:
print("CNN simples")
print(classification_report(test_targets, cnn_predictions, target_names=CLASS_NAMES, zero_division=0))
print("ResNet-18")
print(classification_report(test_targets, resnet_predictions, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for axis, predictions, title in [
    (axes[0], cnn_predictions, "CNN simples"),
    (axes[1], resnet_predictions, "ResNet-18"),
]:
    matrix = confusion_matrix(test_targets, predictions)
    sns.heatmap(
        matrix, annot=True, fmt="d", cmap="Blues", ax=axis,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    )
    axis.set_title(title)
    axis.set_xlabel("Predito")
    axis.set_ylabel("Real")
    axis.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## Exemplo de inferência

As probabilidades abaixo são scores produzidos pelo modelo. Elas não foram calibradas e não representam confiança clínica.

In [ ]:
sample_index = 0
sample_record = test_frame.iloc[sample_index]
sample_tensor, sample_target = test_dataset[sample_index]

resnet.eval()
with torch.no_grad():
    sample_logits = resnet(sample_tensor.unsqueeze(0).to(device))
    sample_probabilities = torch.softmax(sample_logits, dim=1).squeeze(0).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
image = Image.open(DATA_DIR / sample_record["relative_path"]).convert("L")
axes[0].imshow(image, cmap="gray")
axes[0].set_title(f"Classe real: {CLASS_NAMES[sample_target.item()]}")
axes[0].axis("off")
sns.barplot(x=sample_probabilities, y=CLASS_NAMES, ax=axes[1])
axes[1].set_xlim(0, 1)
axes[1].set_xlabel("Probabilidade produzida pelo modelo")
plt.tight_layout()
plt.show()

## Conclusões e limitações

Nesta aula vimos que:

- convoluções aprendem filtros locais e preservam relações espaciais;
- uma CNN pode aprender features diretamente dos pixels;
- conexões residuais facilitam o treinamento de redes profundas;
- transfer learning reutiliza representações aprendidas em outro dataset;
- congelar o backbone é diferente de fazer fine-tuning;
- uma arquitetura maior não garante desempenho superior.

As métricas devem ser interpretadas com cautela: o dataset é pequeno e desbalanceado, não há identificação de pacientes para um split por paciente, e as imagens não representam uma validação clínica prospectiva.